[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nishidaab26002-arch/sake-review-analysis/blob/main/programs/03_parse_review_text.ipynb)

# 03. 口コミ文字列の構造化

スクレイピングで取得した口コミは、ユーザー名・評価・本文・スペック情報・投稿日が**1本の文字列**に連結されています。
正規表現で各項目に分解し、分析に使える表形式にします。


In [1]:
# --- 実行環境の設定 ---------------------------------------------------------
# Colab: Google Driveをマウントし、BASE_DIR 配下を読み書きします。
# ローカル: 環境変数 SAKE_PROJECT_DIR でデータ置き場を指定できます。
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/作成したプログラム等/日本酒口コミ分析'  # 自分のDriveのパスに変更
except ImportError:
    BASE_DIR = os.environ.get('SAKE_PROJECT_DIR', './sake_review_analysis')

CSV_DIR = f'{BASE_DIR}/口コミ_CSV'          # 収集・加工したCSV
for d in (CSV_DIR,):
    os.makedirs(d, exist_ok=True)

Mounted at /content/drive


# 実行

In [3]:
import pandas as pd
import re

# 入出力パスの設定（元のプレーンデータから再スタートします）
INPUT_PATH = f'{CSV_DIR}/saketime_robust_result.csv'
OUTPUT_PATH = f'{CSV_DIR}/saketime_analyzed.csv'

def extract_specs_backward(text):
    """
    文字列の後ろから順番に確定キーワードを探し、切り離していく抽出パイプライン
    """
    # 初期値
    extracted = {
        "user_name": "不明",
        "rating": "不明",
        "review_body": "不明",
        "specific_name": "不明",
        "sake_type": "不明", # 生酒などの「酒の種類」も追加抽出
        "rice_cultivar": "不明",
        "body_type": "不明",
        "amakara": "不明",
        "post_date": "不明"
    }

    current_text = str(text)

    # --------------------------------------------------
    # Step 1: 【日付】を末尾から切り離す
    # --------------------------------------------------
    date_match = re.search(r'(\d{4}年\d{1,2}月\d{1,2}日)$', current_text)
    if date_match:
        extracted["post_date"] = date_match.group(1)
        current_text = current_text[:date_match.start()].strip()

    # --------------------------------------------------
    # Step 2: 【甘辛度】を末尾から切り離す
    # --------------------------------------------------
    amakara_match = re.search(r'甘辛:(.*?)$', current_text)
    if amakara_match:
        extracted["amakara"] = amakara_match.group(1).strip()
        current_text = current_text[:amakara_match.start()].strip()

    # --------------------------------------------------
    # Step 3: 【ボディ感】を末尾から切り離す
    # --------------------------------------------------
    # 「テイストボディ:」または「ボディ:」に対応
    body_match = re.search(r'(?:テイスト)?ボディ:(.*?)$', current_text)
    if body_match:
        extracted["body_type"] = body_match.group(1).strip()
        current_text = current_text[:body_match.start()].strip()
        # 万が一「テイスト」という文字だけが残ってしまったら除去
        current_text = re.sub(r'テイスト$', '', current_text).strip()

    # --------------------------------------------------
    # Step 3.5: 【酒の種類】（生酒など）があれば切り離す（特定名称への混入防止）
    # --------------------------------------------------
    sake_type_match = re.search(r'酒の種類(.*?)$', current_text)
    if sake_type_match:
        extracted["sake_type"] = sake_type_match.group(1).strip()
        current_text = current_text[:sake_type_match.start()].strip()


    # --------------------------------------------------
    # Step 4: 【原料米】を末尾から切り離す（文字数ガード付き）
    # --------------------------------------------------
    rice_match = re.search(r'原料米(.*?)$', current_text)
    if rice_match:
        # 一旦、マッチした部分の文字列を rice_cultivar に入れる
        extracted["rice_cultivar"] = rice_match.group(1).strip()

        # 🌾 その文字数が10文字以内（本物のスペックの可能性が高い）場合のみ、文頭を削る
        if len(extracted["rice_cultivar"]) <= 10:
            current_text = current_text[:rice_match.start()].strip()
        else:
            # 10文字を超える場合は、本文中の「原料米」という単語に誤反応したとみなす
            # 取得結果を「不明」に戻し、文頭（本文）も削らずにスルーする
            extracted["rice_cultivar"] = "不明"



    # --------------------------------------------------
    # Step 5: 【特定名称】を末尾から切り離す
    # --------------------------------------------------
    spec_match = re.search(r'特定名称(.*?)$', current_text)
    if spec_match:
        extracted["specific_name"] = spec_match.group(1).strip()
        current_text = current_text[:spec_match.start()].strip()

    # --------------------------------------------------
    # Step 6: 残った先頭部分から【ユーザー名】【評価点】【本文】を分離
    # --------------------------------------------------
    # ユーザー名と点数 (例: "ユーザー名4.5" を分離)
    user_match = re.search(r'^(.*?)([0-9]\.[0-9]|-)', current_text)
    if user_match:
        extracted["user_name"] = user_match.group(1).strip()
        # 評価点が数字（4.5など）であれば取得、ハイフンなどの未評価記号なら不明へ
        rating_str = user_match.group(2).strip()
        extracted["rating"] = rating_str if rating_str != '-' else "不明"

        # 残ったすべてが純粋な口コミ本文
        extracted["review_body"] = current_text[user_match.end():].strip()
    else:
        # 万が一ユーザー名や点数が見つからなかった場合はそのまま本文に
        extracted["review_body"] = current_text

    return extracted

# =========================================================
# 実行処理
# =========================================================
print("⚙️ 最適化された抽出パイプラインを起動します...")

# 1. プレーンな取得結果データを読み込む
df = pd.read_csv(INPUT_PATH)
total_rows = len(df)

# 2. 関数を適用して分解
extracted_data = df['review'].apply(extract_specs_backward)
df_extracted = pd.DataFrame(extracted_data.tolist())

# 3. 元のデータ（prefecture, brand, maker, review_num, pricerange）と結合
df_final = pd.concat([df.drop(columns=['review']), df_extracted], axis=1)

# 4. 保存
df_final.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')

# =========================================================
# 最終的なデータ品質（欠損率）のレポート
# =========================================================
print(f"\n✅ 抽出完了！新しいファイルを作成しました: {OUTPUT_PATH}")
print(f"\n📊 --- [構造化データ 品質レポート (全{total_rows}件)] ---")
report_cols = ['specific_name', 'rice_cultivar', 'body_type', 'amakara', 'sake_type']

for col in report_cols:
    valid_count = (df_final[col] != '不明').sum()
    missing_count = total_rows - valid_count
    missing_ratio = (missing_count / total_rows) * 100
    print(f" 🔹 {col:<15} : 取得済 {valid_count:3}件 / 不明 {missing_count:3}件 (欠損率 {missing_ratio:.1f}%)")

print("\n💡 分析（Doc2Vec等）への活用準備が完全に整いました。")

⚙️ 最適化された抽出パイプラインを起動します...

✅ 抽出完了！新しいファイルを作成しました: /content/drive/MyDrive/作成したプログラム等/日本酒口コミ分析/口コミ_CSV/saketime_analyzed.csv

📊 --- [構造化データ 品質レポート (全10847件)] ---
 🔹 specific_name   : 取得済 5701件 / 不明 5146件 (欠損率 47.4%)
 🔹 rice_cultivar   : 取得済 3615件 / 不明 7232件 (欠損率 66.7%)
 🔹 body_type       : 取得済 6786件 / 不明 4061件 (欠損率 37.4%)
 🔹 amakara         : 取得済 6855件 / 不明 3992件 (欠損率 36.8%)
 🔹 sake_type       : 取得済 3881件 / 不明 6966件 (欠損率 64.2%)

💡 分析（Doc2Vec等）への活用準備が完全に整いました。
